# The catalogue, fragmented

What BRICS produced from the **Enamine REAL lead-like** catalogue: how many
fragments and bonded pairs came out, how big they are, and how often any one of
them is reused. The DEL notebooks (`01_fragment_report.ipynb`,
`02_combination_report.ipynb`) rank fragments by enrichment; nothing here is
ranked, because the catalogue carries no activity labels. This is an inventory.

**Where each number comes from, and how far it can be trusted**

| source | what it is | complete? |
|---|---|---|
| `work_catalogue/fragments.parquet` | every distinct fragment, with how many compounds carry it | **yes** — the fragment pass ran over all 9,857 input files |
| `work_catalogue/compound_fragments/` | per compound, its fragment ids and the bonds between them | **yes**, but 24.6 billion rows — anything taken from it here is a shard sample |
| `work_catalogue/combination_counts/size2/` | bonded pairs the pass was *told to look for* | **no** — see below |
| `work_catalogue_overlap/combination_counts/size2/` | a second pass against a uniform sample of DEL pairs | **no** — same reason |

The combination stage never enumerated the catalogue's own pairs. It was handed a
target list of DEL pairs and asked which of them the catalogue also has, which is
the right question for transferring DEL hits and the wrong one for counting what
the catalogue contains. So **the catalogue's pair vocabulary is measured here by
sampling `compound_fragments` directly**, and the two target-list passes are
reported separately, for what they actually answer.


In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "notebooks"))
sys.path.insert(0, str(Path.cwd().parent))
from report_style import (SERIES, INK_SOFT, use_report_style, strip_spines,
                          caption, stat_tiles, draw_grid)

use_report_style()

WORK = Path("../work_catalogue")              # the catalogue fragmentation run
OVERLAP = Path("../work_catalogue_overlap")   # the second, unbiased pair pass
DEL_WORK = Path("../work_full")               # for the side-by-side at the end
SIZE = 2                                      # combination size
SAMPLE_SHARDS = 16                            # shards read where no table has the answer
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
# Tables below mix billions with ratios in one column; without this pandas
# renders the whole column in scientific notation.
pd.set_option("display.float_format",
              lambda v: f"{v:,.0f}" if abs(v) >= 1000 else f"{v:,.2f}")
print("reading from", WORK.resolve())


In [ ]:
frag_summary = json.loads((WORK / "summary_fragment.json").read_text())
combo_summary = json.loads((WORK / "summary_combine.json").read_text())
N_COMPOUNDS = int(frag_summary["n_compounds"])
MIN_HAC = int(frag_summary["min_hac"])

frags = pq.read_table(WORK / "fragments.parquet",
                      columns=["frag_id", "n_compounds", "hac", "n_attachments"]).to_pandas()
occ = frags["n_compounds"].to_numpy()
INSTANCES = int(occ.sum())

print(f"{N_COMPOUNDS:,} compounds fragmented at --min-hac {MIN_HAC}, "
      f"over {int(frag_summary['n_files']):,} input files")
print(f"{len(frags):,} distinct fragments")
print(f"{INSTANCES:,} compound-fragment instances "
      f"({INSTANCES / N_COMPOUNDS:.2f} fragments per compound)")

# A fragment with no attachment point is a whole molecule that came through
# uncut.  These two numbers are easy to confuse and are three orders of
# magnitude apart, so both are computed here and used everywhere below.
uncut = frags["n_attachments"] == 0
UNCUT_TYPES_PCT = 100 * float(uncut.mean())
UNCUT_COMPOUND_PCT = 100 * float(occ[uncut.to_numpy()].sum()) / N_COMPOUNDS
print(f"\nnever cut: {UNCUT_TYPES_PCT:.1f}% of the fragment vocabulary, "
      f"but only {UNCUT_COMPOUND_PCT:.4f}% of compounds "
      f"(1 in {N_COMPOUNDS / occ[uncut.to_numpy()].sum():,.0f})")
frags.head(3)


## Headline numbers


In [ ]:
fig = stat_tiles([
    ("compounds fragmented", f"{N_COMPOUNDS / 1e9:.2f}B"),
    ("distinct fragments", f"{len(frags):,}"),
    ("fragment instances", f"{INSTANCES / 1e9:.1f}B"),
    ("fragments per compound", f"{INSTANCES / N_COMPOUNDS:.2f}"),
    ("seen in 10+ compounds", f"{int((occ >= 10).sum()):,}"),
    ("seen in one compound only", f"{(occ == 1).mean():.0%}"),
    ("never cut: share of the vocabulary", f"{UNCUT_TYPES_PCT:.0f}%"),
    ("never cut: share of compounds", f"{UNCUT_COMPOUND_PCT:.3f}%"),
])
plt.show()


## How big are the fragments?

Two readings of the same column, and they disagree on purpose. Counting each
fragment once describes the *vocabulary*; weighting by how many compounds carry
it describes what you actually meet walking through the catalogue. The vocabulary
is dominated by rare large fragments, the usage by a few small ones.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharex=True)
bins = np.arange(frags.hac.min() - 0.5, frags.hac.max() + 1.5)

axes[0].hist(frags.hac, bins=bins, color=SERIES[0], linewidth=0)
axes[0].set_ylabel("distinct fragments")
axes[0].set_title("Fragment size, one count per fragment")

axes[1].hist(frags.hac, bins=bins, weights=occ / INSTANCES, color=SERIES[1], linewidth=0)
axes[1].set_ylabel("share of all instances")
axes[1].set_title("...weighted by how often it is used")

for ax in axes:
    ax.axvline(MIN_HAC - 0.5, color=INK_SOFT, lw=1.2, ls="--")
    ax.set_xlabel("heavy atoms")
    strip_spines(ax)
fig.tight_layout()

caption(axes[0], f"Dashed line is the --min-hac {MIN_HAC} floor: a cut that would leave anything "
                 "smaller is undone, so nothing exists below it.\n"
                 f"The median fragment is {frags.hac.median():.0f} heavy atoms counted by "
                 f"vocabulary and {np.average(frags.hac, weights=occ):.1f} counted by usage - the "
                 "big ones are the rare ones.")
plt.show()


## What BRICS could and could not cut

`n_attachments` is how many open attachment points a fragment carries — the same
as how many BRICS bonds were broken around it. **Zero means the whole molecule
came through intact.**

> **Read the next number carefully.** Uncut molecules are **40% of the fragment
> vocabulary but 0.038% of the compounds** — 9.3 million occurrences out of 24.6
> billion, about 1 compound in 2,647. They are nearly all singletons, so they
> inflate the dictionary enormously while costing almost no coverage. A claim
> that "40% of the catalogue does not get fragmented" would be wrong by three
> orders of magnitude.

And it is mostly not BRICS's fault. Refragmenting 20,000 of these molecules
(`fragmentation_test/`) shows **91.5% of them do have BRICS bonds** — the
`--min-hac 6` floor undid every cut, because no split left both halves at six
heavy atoms. Only 8.5% have no cuttable bond at all. The floor, not the rule set,
is what produces this 40%.


In [ ]:
by_att = frags.assign(singleton=occ == 1).groupby("n_attachments").agg(
    types=("frag_id", "size"),
    instances=("n_compounds", "sum"),
    median_hac=("hac", "median"),
    singleton_pct=("singleton", "mean"),
)
by_att["pct_of_types"] = 100 * by_att["types"] / len(frags)
by_att["pct_of_instances"] = 100 * by_att["instances"] / INSTANCES
by_att["singleton_pct"] = 100 * by_att["singleton_pct"]
by_att[["types", "pct_of_types", "instances", "pct_of_instances",
        "median_hac", "singleton_pct"]].round(2)


In [ ]:
index = np.arange(len(by_att))
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(index - 0.2, by_att["pct_of_types"], width=0.4, color=SERIES[0],
       linewidth=0, label="share of distinct fragments")
ax.bar(index + 0.2, by_att["pct_of_instances"], width=0.4, color=SERIES[1],
       linewidth=0, label="share of fragment instances")
ax.set_xticks(index)
ax.set_xticklabels(by_att.index)
ax.set_xlabel("attachment points (0 = the molecule was never cut)")
ax.set_ylabel("per cent")
ax.set_title("Vocabulary and usage, split by how many cuts were made")
ax.legend(loc="upper right")
strip_spines(ax)
uncut = by_att.loc[0]
caption(ax, f"Uncut whole molecules are {uncut['pct_of_types']:.0f}% of the vocabulary but only "
            f"{uncut['pct_of_instances']:.2f}% of the usage, and {uncut['singleton_pct']:.1f}% of "
            "them are seen in a single compound. Vocabulary and coverage are not the same thing.")
plt.show()


## Why are some fragments so large?

Fragments run to 30 heavy atoms here and 33 in the DEL, which looks like a
failure of the cutting. It is the same type-versus-usage trap as above: **43% of
catalogue fragment *types* are 18 heavy atoms or more, but they account for
0.105% of fragment *instances*.*** Large fragments are rare in use and
overwhelming in the dictionary.

The cause differs between the two libraries, and so does the fix.


In [ ]:
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

ALKYNE = Chem.MolFromSmarts("[CX2]#[CX2]")
rng = np.random.default_rng(0)
rows = []
for name, work in [("catalogue", WORK), ("DEL", DEL_WORK)]:
    table = ds.dataset(work / "fragments.parquet").to_table(
        columns=["frag_smiles", "hac", "n_attachments", "n_compounds"]).to_pandas()
    counts = table["n_compounds"].to_numpy()
    big = table["hac"] >= 18
    for label, selected in [("18+ heavy atoms", big), ("under 18", ~big)]:
        subset = table[selected]
        drawn = subset.iloc[rng.choice(len(subset), min(20_000, len(subset)), replace=False)]
        alkyne = np.mean([
            (mol := Chem.MolFromSmiles(s)) is not None and mol.HasSubstructMatch(ALKYNE)
            for s in drawn["frag_smiles"]])
        rows.append({
            "library": name, "fragments": label,
            "% of types": round(100 * float(selected.mean()), 2),
            "% of instances": round(100 * counts[selected.to_numpy()].sum() / counts.sum(), 3),
            "% uncut whole molecules": round(100 * float((drawn["n_attachments"] == 0).mean()), 1),
            "% containing an alkyne": round(100 * float(alkyne), 1),
        })
pd.DataFrame(rows).set_index(["library", "fragments"])


Two different problems wearing the same face:

* **In the catalogue, large fragments are mostly not fragments at all** — 78% of
  them are whole molecules that were never cut, so this is the `--min-hac` story
  from the section above, seen from the other end. Alkynes are no more common in
  the large ones than the small ones (2.8% against 3.0%): irrelevant here.
* **In the DEL, they are real fragments that BRICS cannot split.** Only 4% are
  uncut molecules, but **67% contain a carbon–carbon triple bond** against 8.9%
  of the small ones. **The alkyne is not in the BRICS bond set** — there is no
  rule permitting a cut next to one — so an alkyne-linked arm stays attached to
  whatever it is joined to, however big that makes the fragment.

The fix follows the cause, and both were measured in `fragmentation_test/`
(100,000 molecules per library, at the production `--min-hac 6`):

| | fragments 18+ heavy atoms | |
|---|---|---|
| | **DEL** | **catalogue** |
| BRICS, as the pipeline runs it | 2.34% | 0.11% |
| BRICS + alkyne, biaryl and ether rules | **0.06%** | 0.11% |
| RECAP | 3.44% | 0.94% |
| ring–linker | 1.39% | 1.90% |

Adding three bond rules cuts the DEL's oversized fragments by **39×** and leaves
the catalogue untouched, exactly as the diagnosis predicts. It also shrinks the
DEL's sampled vocabulary from 29,900 fragments to 20,762 while dropping the
seen-once rate from 51.5% to 29.6% — the alkyne-bearing giants were nearly all
unique, and cutting them yields pieces that recur.

The catalogue's large fragments need the other lever: a lower floor, or a cut
selector that does not cascade a molecule back to one piece. See
`fragmentation_test/README.md`.


## How often is a fragment reused?

The question the catalogue has to answer before it is useful for transfer: a
fragment that appears in one compound is not a building block, it is that
compound.


In [ ]:
rows = []
for threshold in [1, 2, 5, 10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    selected = occ >= threshold
    rows.append({
        "in at least N compounds": threshold,
        "fragments": int(selected.sum()),
        "% of fragments": round(100 * selected.mean(), 2),
        "% of all instances": round(100 * occ[selected].sum() / INSTANCES, 3),
    })
reuse = pd.DataFrame(rows)
reuse


In [ ]:
ranked = np.sort(occ)[::-1]
coverage = np.cumsum(ranked, dtype=np.float64) / INSTANCES
rank = np.arange(1, len(ranked) + 1)
# Thinned only for drawing, log-spaced so the head of the curve keeps its shape.
picks = np.unique(np.concatenate([
    np.arange(min(2_000, len(ranked))),
    (np.geomspace(1, len(ranked), 6_000) - 1).astype(np.int64),
]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].plot(rank[picks], ranked[picks], color=SERIES[0])
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("fragment rank")
axes[0].set_ylabel("compounds containing it")
axes[0].set_title("Rank against frequency")

axes[1].plot(rank[picks], 100 * coverage[picks], color=SERIES[0])
axes[1].set_xscale("log")
axes[1].set_xlabel("fragments taken, most common first")
axes[1].set_ylabel("per cent of all instances covered")
axes[1].set_title("How few fragments account for most of the catalogue")
for level in (0.5, 0.9, 0.99):
    k = int(np.searchsorted(coverage, level)) + 1
    axes[1].plot([k, k], [0, 100 * level], color=INK_SOFT, lw=0.9, ls="--")
    axes[1].annotate(f"{level:.0%} at {k:,}", xy=(k, 100 * level), xytext=(6, -12),
                     textcoords="offset points", color=INK_SOFT, fontsize=9)

for ax in axes:
    strip_spines(ax)
fig.tight_layout()

half = int(np.searchsorted(coverage, 0.5)) + 1
caption(axes[0], f"{(occ == 1).mean():.0%} of fragments are seen once; the flat tail on the "
                 "right is almost the whole vocabulary.")
caption(axes[1], f"The {half:,} most common fragments - {100 * half / len(occ):.3f}% of them - "
                 "cover half of every fragment instance in the catalogue.")
plt.show()


## The most reused fragments

Read straight off the table, so these are exact counts over all 24.6 billion
compounds.


In [ ]:
wanted = frags.nlargest(25, "n_compounds")["frag_id"].tolist()
top = ds.dataset(WORK / "fragments.parquet").to_table(
    columns=["frag_id", "frag_name", "frag_smiles", "hac", "n_attachments", "n_compounds"],
    filter=pc.field("frag_id").isin(wanted),
).to_pandas().sort_values("n_compounds", ascending=False).reset_index(drop=True)
top["% of compounds"] = (100 * top["n_compounds"] / N_COMPOUNDS).round(2)
top[["frag_name", "frag_smiles", "hac", "n_attachments", "n_compounds", "% of compounds"]]


In [ ]:
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

mols, legends = [], []
for _, row in top.head(12).iterrows():
    mol = Chem.MolFromSmiles(row.frag_smiles)
    if mol is None:
        continue
    mols.append(mol)
    legends.append(f"{row.frag_name}   {row.n_compounds / 1e6:,.0f}M compounds\n"
                   f"{row['% of compounds']:.1f}% of the catalogue")
draw_grid(mols, legends, mols_per_row=4, sub_img_size=(300, 230))


## How many fragments does a compound break into?

In no summary table, so it is read off a sample of shards. A compound fragments
into a tree, so one with *n* fragments contributes exactly *n* − 1 bonded pairs —
which is where the next section's numbers come from.


In [ ]:
MIX_A = np.uint64(0x9E3779B97F4A7C15)
MIX_B = np.uint64(0x165667B19E3779F9)
MIX_C = np.uint64(0xBF58476D1CE4E5B9)


def _flat(column):
    # One contiguous Arrow array from a possibly chunked column.
    array = column.combine_chunks()
    return array.chunk(0) if hasattr(array, "chunk") else array


def shard_stats(path):
    # Fragments per compound, and a 64-bit key per bonded pair, for one shard.
    #
    # Fragment ids are already 64-bit content hashes, so a pair of them does not
    # fit in one word; mixing them down to a single integer is what keeps
    # `unique` and `sort` affordable at this size. Collisions are possible in
    # principle and negligible here - a few times 10^7 keys against 2^64.
    table = pq.read_table(path, columns=["frag_ids", "edge_src", "edge_dst"])
    ids = _flat(table["frag_ids"])
    offsets = ids.offsets.to_numpy().astype(np.int64)
    values = ids.values.to_numpy()

    src = _flat(table["edge_src"])
    base = np.repeat(offsets[:-1], np.diff(src.offsets.to_numpy().astype(np.int64)))
    a = values[base + src.values.to_numpy().astype(np.int64)]
    b = values[base + _flat(table["edge_dst"]).values.to_numpy().astype(np.int64)]

    low = np.minimum(a, b).astype(np.uint64)
    high = np.maximum(a, b).astype(np.uint64)
    key = low * MIX_A ^ (high + MIX_B)
    key ^= key >> np.uint64(31)
    key *= MIX_C
    key ^= key >> np.uint64(27)
    return np.diff(offsets), key


shards = sorted((WORK / "compound_fragments").glob("*.parquet"))
rng = np.random.default_rng(0)
sample = [shards[i] for i in
          np.sort(rng.choice(len(shards), min(SAMPLE_SHARDS, len(shards)), replace=False))]

size_histogram = np.zeros(24, dtype=np.int64)
unique_keys, key_counts, first_shard_of_key, compounds_per_shard = [], [], [], []
pair_instances = 0
for position, path in enumerate(sample):
    sizes, keys = shard_stats(path)
    size_histogram += np.bincount(np.clip(sizes, 0, 23), minlength=24)
    distinct, counts = np.unique(keys, return_counts=True)
    unique_keys.append(distinct)
    key_counts.append(counts.astype(np.int32))
    first_shard_of_key.append(np.full(len(distinct), position, dtype=np.int32))
    compounds_per_shard.append(len(sizes))
    pair_instances += len(keys)

SAMPLED_COMPOUNDS = int(size_histogram.sum())
print(f"{len(sample)} shards, {SAMPLED_COMPOUNDS:,} compounds "
      f"({100 * SAMPLED_COMPOUNDS / N_COMPOUNDS:.3f}% of the catalogue)")


In [ ]:
present = np.flatnonzero(size_histogram)
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(present, 100 * size_histogram[present] / SAMPLED_COMPOUNDS,
       color=SERIES[0], width=0.8, linewidth=0)
ax.set_xlabel("fragments the compound breaks into")
ax.set_ylabel("per cent of compounds")
ax.set_title("Fragments per compound")
strip_spines(ax)
mean_size = float(np.average(present, weights=size_histogram[present]))
caption(ax, f"Mean {mean_size:.2f} fragments per compound, so {mean_size - 1:.2f} bonded pairs. "
            f"Only {100 * size_histogram[1] / SAMPLED_COMPOUNDS:.2f}% of compounds survive whole.")
plt.show()


---

# Bonded pairs

Two fragments joined at the bond BRICS broke between them. The sampled shards
above already carry every pair inside them, so the catalogue's own pair
vocabulary is measured from that sample rather than from the target-list passes.


In [ ]:
keys = np.concatenate(unique_keys)
counts = np.concatenate(key_counts)
origin = np.concatenate(first_shard_of_key)
order = np.argsort(keys, kind="stable")
keys, counts, origin = keys[order], counts[order], origin[order]

starts = np.flatnonzero(np.r_[True, keys[1:] != keys[:-1]])
pair_occurrences = np.add.reduceat(counts.astype(np.int64), starts)
first_shard = np.minimum.reduceat(origin, starts)      # stable sort keeps shard order
discovered = np.cumsum(np.bincount(first_shard, minlength=len(sample)))
scanned = np.cumsum(compounds_per_shard)
DISTINCT_PAIRS = int(len(starts))

del keys, counts, origin, order
print(f"{SAMPLED_COMPOUNDS:,} compounds carry {pair_instances:,} pair instances "
      f"({pair_instances / SAMPLED_COMPOUNDS:.2f} each)")
print(f"{DISTINCT_PAIRS:,} distinct pairs in the sample")
print(f"{100 * (pair_occurrences == 1).mean():.1f}% of them were seen exactly once")


### Does the pair vocabulary run out?

If the catalogue drew its pairs from a small set this curve would bend over. The
slope at the right-hand end says whether the sample has seen the vocabulary or
merely started on it.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(scanned / 1e6, discovered / 1e6, color=SERIES[0], marker="o", ms=4)
ax.set_xlabel("compounds scanned (millions)")
ax.set_ylabel("distinct pairs found (millions)")
ax.set_title("New pairs keep arriving")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
strip_spines(ax)
last_slope = (discovered[-1] - discovered[-2]) / (scanned[-1] - scanned[-2])
caption(ax, f"The last shard still brought {last_slope:.2f} new pairs per compound scanned. "
            f"{DISTINCT_PAIRS:,} pairs out of {100 * SAMPLED_COMPOUNDS / N_COMPOUNDS:.3f}% of the "
            "catalogue is a floor, not a count.")
plt.show()


In [ ]:
rows = []
for threshold in [1, 2, 5, 10, 100, 1_000]:
    selected = pair_occurrences >= threshold
    rows.append({
        "in at least N sampled compounds": threshold,
        "pairs": int(selected.sum()),
        "% of sampled pairs": round(100 * selected.mean(), 2),
        "% of pair instances": round(100 * pair_occurrences[selected].sum() / pair_instances, 2),
    })
pd.DataFrame(rows)


### What the two target-list passes found

Neither pass counts the catalogue's pairs; both ask *which DEL pairs the
catalogue also has*. They differ in which DEL pairs they were given, and that
difference is the whole point:

* **binder-conditioned** — the 64,693 pairs carried by at least one DEL binder.
  Those sit in chemistry the DEL was selected for, so the hit rate is inflated.
* **uniform** — a 1-in-200 sample of distinct DEL pairs regardless of activity,
  which is the unbiased figure.

The gap between the two is a statement about the DEL, not about the catalogue.


In [ ]:
def pair_table(path):
    # Distinct pairs and the compounds behind them, for one target-list pass.
    # Reads 9,857 files, so this cell takes about a minute.
    frame = ds.dataset(path).to_table(
        columns=["frag_0", "frag_1", "n_compounds"]).to_pandas()
    return frame.groupby(["frag_0", "frag_1"], sort=False)["n_compounds"].sum()


passes = []
for label, work in [("binder-conditioned", WORK), ("uniform 1-in-200", OVERLAP)]:
    searched = int(json.loads((work / "summary_combine.json").read_text())["n_target_combos"])
    found = pair_table(work / "combination_counts" / f"size{SIZE}")
    passes.append({
        "pass": label,
        "DEL pairs searched for": searched,
        "found in the catalogue": int(len(found)),
        "% found": round(100 * len(found) / searched, 2),
        "catalogue compounds carrying one": int(found.sum()),
        "median compounds per found pair": int(found.median()),
    })
pd.DataFrame(passes).set_index("pass")


---

## The catalogue against the DEL

Same fragmentation settings on both, so the two inventories compare directly. The
catalogue is about ten times the compounds and twenty-seven times the fragment
vocabulary — but far more of that vocabulary is dead weight.


In [ ]:
def inventory(work):
    summary = json.loads((work / "summary_fragment.json").read_text())
    table = pq.read_table(work / "fragments.parquet",
                          columns=["n_compounds", "hac", "n_attachments"]).to_pandas()
    counts = table["n_compounds"].to_numpy()
    total = counts.sum()
    ranked = np.cumsum(np.sort(counts)[::-1], dtype=np.float64) / total
    return {
        "compounds": int(summary["n_compounds"]),
        "distinct fragments": len(table),
        "fragment instances": int(total),
        "fragments per compound": round(total / int(summary["n_compounds"]), 2),
        "median heavy atoms": int(table["hac"].median()),
        "seen once only, %": round(100 * (counts == 1).mean(), 1),
        "in 10+ compounds": int((counts >= 10).sum()),
        "in 10+ compounds, %": round(100 * (counts >= 10).mean(), 1),
        "never cut, % of types": round(100 * (table["n_attachments"] == 0).mean(), 1),
        # Per million, because as a percentage this rounds to zero and the
        # whole point of the row is that it is tiny next to the one above.
        "never cut, compounds per million": round(
            1e6 * counts[(table["n_attachments"] == 0).to_numpy()].sum()
            / int(summary["n_compounds"]), 1),
        "18+ heavy atoms, % of types": round(100 * (table["hac"] >= 18).mean(), 1),
        "fragments covering half the instances": int(np.searchsorted(ranked, 0.5)) + 1,
    }


comparison = pd.DataFrame({"catalogue": inventory(WORK), "DEL": inventory(DEL_WORK)})
ratio = comparison["catalogue"] / comparison["DEL"].replace(0, np.nan)
comparison["catalogue / DEL"] = ratio.round(2)
comparison


The last two rows are the ones that matter for transfer. The catalogue carries
27× the DEL's fragment vocabulary, but needs only 3.3× as many fragments to cover
half its usage and has only 8× as many clearing the ten-compound bar.

Most of the extra vocabulary is dead weight: 61% of catalogue fragments are seen
in a single compound against 12% in the DEL, and 40% of its *types* are whole
molecules BRICS never cut against 1.4%. Note the two "never cut" rows — 40% of
types is 378 compounds per million, and only the first of those is a large
number. A
catalogue ten times the size is not a proportionally bigger set of usable
building blocks.


## Export


In [ ]:
out = Path("../reports")
out.mkdir(exist_ok=True)
reuse.to_csv(out / "catalogue_fragment_reuse.csv", index=False)
comparison.to_csv(out / "catalogue_vs_del_inventory.csv")
top[["frag_name", "frag_smiles", "hac", "n_attachments", "n_compounds"]].to_csv(
    out / "catalogue_top_fragments.csv", index=False)
(out / "catalogue_summary.json").write_text(json.dumps({
    "compounds": N_COMPOUNDS,
    "min_hac": MIN_HAC,
    "distinct_fragments": int(len(frags)),
    "fragment_instances": INSTANCES,
    "fragments_in_10_or_more_compounds": int((occ >= 10).sum()),
    "fragments_seen_once_pct": round(100 * float((occ == 1).mean()), 2),
    "never_cut_pct": round(100 * float((frags.n_attachments == 0).mean()), 2),
    "sampled_compounds": SAMPLED_COMPOUNDS,
    "distinct_pairs_in_sample": DISTINCT_PAIRS,
    "pair_instances_in_sample": int(pair_instances),
}, indent=2) + "\n")
print("wrote", out.resolve())
